# NLP Fine-Tuning Chatbot Project

In this notebook, you will:
1. Try the **base model** (before fine-tuning) to see how it responds
2. **Fine-tune** the model on a dataset of your choice
3. Chat with the **fine-tuned model** and compare the difference
4. Run **multiple experiments** and compare results side by side

```
                         config.yaml
                             |
                             v
    datasets/*.json --> finetune.py --> output/
                             |              |
                             v              v
                      experiments/       app.py --> Gradio Chatbot
                     (before/after)       (shareable link)
```

**Just run each cell in order!** The defaults (TinyLlama + QA dataset + LoRA) work out of the box.

---
## Step 1: Setup

**Choose ONE of the two options below** depending on where you are running this notebook.

### Option A: Google Colab Setup

Run the two cells below to clone the repo and install dependencies.

**Skip this section if you are running locally.**

In [ ]:
# ============================================
# COLAB ONLY — Clone the project from GitHub
# ============================================
!git clone https://github.com/ratulalahy/nlp_course_sample_chat_fine_tune.git
%cd nlp_course_sample_chat_fine_tune

In [ ]:
# ============================================
# COLAB ONLY — Install dependencies
# ============================================
!pip install -r requirements.txt -q
print("All packages installed!")

### Option B: Local Setup

If you are running this notebook on your own machine, open a terminal and run:

```bash
git clone https://github.com/ratulalahy/nlp_course_sample_chat_fine_tune.git
cd nlp_course_sample_chat_fine_tune
python -m venv venv
source venv/bin/activate        # On Windows: venv\Scripts\activate
pip install -r requirements.txt
jupyter notebook notebook.ipynb
```

Then continue from **Step 2** below.

**Note:** Training on CPU (no GPU) will be slow but works for small models like TinyLlama and Qwen3-0.6B.

---
## Step 2: Check GPU

Make sure you have a GPU available. On Colab, if this says `No GPU`, go to **Runtime > Change runtime type > T4 GPU**.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU available: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("No GPU detected. Training will run on CPU (slower but works).")
    print("If you're on Colab: Runtime > Change runtime type > T4 GPU")
    print("If you're local: this is fine for small models like TinyLlama.")

---
## Step 3: Choose Your Configuration

Edit the settings below to pick your **model** and **dataset**.

The defaults (TinyLlama + QA Bot + LoRA) are a great starting point.

After your first run, come back here, change the settings, and re-run to compare!

In [ ]:
import yaml

# Load the default config
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

# =============================================
# CHANGE THESE SETTINGS TO EXPERIMENT!
# =============================================

# --- Pick a model (uncomment ONE) ---
config["model"]["name"] = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"    # ~700MB, fastest
# config["model"]["name"] = "Qwen/Qwen3-0.6B"                     # ~400MB, ultra-light
# config["model"]["name"] = "HuggingFaceTB/SmolLM2-1.7B-Instruct" # 1.7B
# config["model"]["name"] = "google/gemma-4-e2b"                   # ~2B, newest
# config["model"]["name"] = "microsoft/Phi-3.5-mini-instruct"     # 3.8B

# --- Pick a dataset (uncomment ONE) ---
config["dataset"]["path"] = "datasets/qa_bot.json"         # General Q&A
# config["dataset"]["path"] = "datasets/uvu_bot.json"      # UVU FAQ
# config["dataset"]["path"] = "datasets/cs_assistant.json"  # CS concepts

# --- Pick training method ---
config["training"]["method"] = "lora"   # "lora" (recommended) or "full"

# --- Training settings (defaults work well) ---
config["training"]["epochs"] = 3
config["training"]["batch_size"] = 4
config["training"]["learning_rate"] = 2.0e-4

# Save updated config
with open("config.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Model:   {config['model']['name']}")
print(f"Dataset: {config['dataset']['path']}")
print(f"Method:  {config['training']['method']}")
print(f"Epochs:  {config['training']['epochs']}")
print("\nConfig saved! Ready to go.")

---
## Step 4: Test Dataset Loading
Quick sanity check — make sure your dataset loads correctly before training.

In [ ]:
!python data_utils.py

---
## Step 5: Chat with the Base Model (Before Fine-Tuning)

Before training, let's see how the **base model** responds to questions from your dataset.

The `--base` flag loads the original model from HuggingFace — no fine-tuning applied.

**Try asking it questions from your dataset** and notice how it gives generic or incorrect answers.

**When you're done chatting, click the stop button (square icon) on this cell to continue to the next step.**

In [ ]:
!python app.py --base

---
## Step 6: Fine-Tune the Model

Now let's train the model on your dataset. The script will:
1. Load the base model
2. Test it on 3 sample questions (**baseline** — before training)
3. Train on your dataset
4. Test the same 3 questions again (**post-training** — after training)
5. Save everything to `output/` and `experiments/`

**Expected time:** ~2-5 minutes on a T4 GPU with TinyLlama.

In [ ]:
!python finetune.py

---
## Step 7: Chat with the Fine-Tuned Model

Now let's see the difference! This loads your **fine-tuned** model.

**Try the same questions you asked the base model in Step 5** — you should see better, more relevant answers!

A shareable URL like `https://xxxxx.gradio.live` will appear. Anyone can try your chatbot for 72 hours — great for your portfolio!

**When you're done, click the stop button on this cell to continue.**

In [ ]:
!python app.py

---
## Step 8: Compare Experiments

Each time you run `finetune.py`, the results are saved to `experiments/`. Run this cell to compare all your experiments side by side.

**To run another experiment:** Go back to **Step 3**, change the model/dataset/settings, then re-run Steps 6-7.

In [ ]:
import json
import os

experiments_dir = "experiments"

if not os.path.exists(experiments_dir):
    print("No experiments yet! Run Step 6 (finetune.py) first.")
else:
    experiments = []
    for fname in sorted(os.listdir(experiments_dir)):
        if fname.endswith(".json"):
            with open(os.path.join(experiments_dir, fname)) as f:
                experiments.append(json.load(f))

    # --- Summary Table ---
    print(f"Found {len(experiments)} experiment(s):\n")
    print(f"{'#':<4} {'Model':<35} {'Dataset':<20} {'Method':<8} {'Loss':<8} {'Time':<8}")
    print("-" * 85)

    for i, exp in enumerate(experiments, 1):
        c = exp["config"]
        r = exp["results"]
        model_short = c["model_name"].split("/")[-1][:33]
        dataset_short = os.path.basename(c["dataset"]).replace(".json", "")
        time_str = f"{r['training_time_seconds']:.0f}s"
        print(f"{i:<4} {model_short:<35} {dataset_short:<20} {c['method']:<8} {r['final_loss']:<8} {time_str:<8}")

    # --- Before vs After for the latest experiment ---
    latest = experiments[-1]
    print(f"\n{'=' * 85}")
    print(f"Latest experiment — Before vs After:")
    print(f"{'=' * 85}")

    for b, a in zip(latest["baseline_responses"], latest["finetuned_responses"]):
        print(f"\nQ: {b['question']}")
        print(f"  BEFORE: {b['response'][:150]}")
        print(f"  AFTER:  {a['response'][:150]}")

---
## What to Try Next

Here's a suggested order for exploring:

| Round | What to change | What you'll learn |
|-------|---------------|-------------------|
| 1 | Nothing (use defaults) | How the full pipeline works |
| 2 | Switch dataset (e.g., `uvu_bot.json`) | How training data affects model behavior |
| 3 | Switch model (e.g., `Qwen/Qwen3-0.6B`) | How model size affects quality |
| 4 | Change epochs (1 vs 3 vs 5) | How training duration affects results |
| 5 | Try `method: "full"` | LoRA vs full fine-tuning (watch GPU memory!) |
| 6 | Create your own dataset | Build a chatbot for any topic you want |

Each run saves to `experiments/` — use Step 8 to compare them all!

For detailed explanations of LoRA, QLoRA, real-world data loading, and portfolio tips, see [GUIDE.md](GUIDE.md).